In [1]:
%pip install ms-fabric-cli --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import notebookutils
import subprocess
import os
import json

In [3]:
must_contain ='DQ'

## CLI Login

In [4]:
# Set environment parameters for Fabric CLI
token = notebookutils.credentials.getToken('pbi')
os.environ['FAB_TOKEN'] = token
os.environ['FAB_TOKEN_ONELAKE'] = token



## Deployment functions

In [5]:
def run_fab_command(command, capture_output=False, silently_continue=False, raw_output=False):
    """
    Executes a Fabric CLI command with optional output capture and error handling.
    """
    result = subprocess.run(["fab", "-c", command], capture_output=capture_output, text=True)
    if not silently_continue and (result.returncode > 0 or result.stderr):
        raise Exception(f"Error running fab command. exit_code: '{result.returncode}'; stderr: '{result}'")
    if capture_output:
        return result if raw_output else result.stdout.strip()
    return None



def List_roles_workspace(workspace_name):
    run_fab_command(f'acl ls {workspace_name}.workspace', capture_output=True, silently_continue=False)
 
def assign_workspace_roles(workspace_name):
    """
    Assigns roles to principals in the workspace.
    """
    workspace_path = f"/{workspace_name}.workspace"

    run_fab_command(f"acl set {workspace_name}.workspace -I 5c906b5c-d1d7-4984-b047-adacd8d795fe -R Member -f",capture_output=True,silently_continue=False)




def cleanup_workspaces(workspace_name):
    run_fab_command(f'rm {workspace_name}.workspace -f ', capture_output=True, silently_continue=True)
    print(f" - Workspace deleted '{workspace_name}'")


def get_workspaces():
    result= run_fab_command("api -X get workspaces/", capture_output=True, silently_continue=True)
    return json.loads(result)["text"]

def filter_items(all_workspaces, must_contain):
    filtered_items = []
    
    for item in all_workspaces["value"]:
        display_name = item.get('displayName', '').lower()
        if must_contain.lower() not in display_name:
            continue  
        filtered_items.append(item["displayName"])

    return filtered_items




In [6]:
all_workspaces = get_workspaces()

In [7]:
workspaces = filter_items(all_workspaces, must_contain)

## Cleanup

In [8]:
workspaces

['DQ1 CODE (D)', 'DQ1 DATA (D)', 'DQ1 CODE (P)', 'DQ1 DATA (P)', 'DQ1 CONFIG']

In [9]:
for displayName in workspaces:
    cleanup_workspaces(displayName)


 - Workspace deleted 'DQ1 CODE (D)'
 - Workspace deleted 'DQ1 DATA (D)'
 - Workspace deleted 'DQ1 CODE (P)'
 - Workspace deleted 'DQ1 DATA (P)'
 - Workspace deleted 'DQ1 CONFIG'
